# 08 — Sintesis de hallazgos e insights (capitulo de resultados)

Este notebook consolida, en lenguaje de tesis, los hallazgos de los notebooks 01 a 07. Es el insumo directo para el capitulo de resultados/discusion del trabajo de grado de Andres Telles y, mas adelante, para el contenido del tablero interactivo de OE-04.

**Alcance de esta sintesis:** unicamente OE-02 (modelo dimensional) y OE-03 (normalizacion, indicadores, EDA e hipotesis), segun el encargo de `PROMPT_EDA_EXTENSO_OE02_OE03.md`. No se incluye OE-01 (extraccion, ya evaluado en `InformesRev/RESUMEN_CODIGO.md`) ni la construccion del tablero interactivo de OE-04 (fase posterior).

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 50)
REPO = Path(".").resolve().parents[1] if Path(".").resolve().name == "notebooks" else Path(".").resolve()
DATA_DIR = REPO / "analitica-descriptiva-andres" / "data"

df = pd.read_parquet(DATA_DIR / "indicadores_restriccion_hierro.parquet")
print(f"Muestra analitica final utilizada en este EDA: {len(df)} contratos de obra publica")
print("(subconjunto de los 48.331 contratos integrados, tras aplicar los criterios de inclusion/exclusion")
print("del anteproyecto — numerales 3.6.3 y 3.6.4 — ver notebook 02).")


Muestra analitica final utilizada en este EDA: 19194 contratos de obra publica
(subconjunto de los 48.331 contratos integrados, tras aplicar los criterios de inclusion/exclusion
del anteproyecto — numerales 3.6.3 y 3.6.4 — ver notebook 02).


## 1. Modelo dimensional (OE-02)

Se construyo un modelo dimensional simplificado sobre archivos Parquet: 1 tabla de hechos (`hechos_contratos`, grano = 1 contrato, 48.331 filas) y 5 tablas de dimension (`dim_proceso`, `dim_proveedor`, `dim_entidad`, `dim_territorio`, `dim_modalidad`), con integridad referencial verificada al 100% en las cinco relaciones. Esto cierra la brecha señalada en `MATRIZ_OBJETIVOS_CODIGO.md` (OE-02, Camino B), documentando explicitamente que es un modelo simplificado sobre Parquet, no un motor relacional formal.

## 2. Calidad de datos y muestra analitica (OE-03)

Se definieron umbrales propios de calidad (apta <=20% nulidad, uso restringido 20-50%, no apta >50%) y se aplicaron por primera vez, de forma verificable, los criterios de inclusion/exclusion 3.6.3/3.6.4 del anteproyecto, incluido el filtro temporal 2018-2024. De los 48.331 contratos de obra integrados, **19.194 (39.7%)** conforman la muestra analitica final (contratos con fecha de firma en 2018-2024, campos minimos completos, valor positivo, fechas coherentes y estado de ejecucion avanzado o finalizado). Se detecto y documento una inconsistencia propia del dato fuente SECOP: 3.538 contratos con `liquidaci_n='Si'` pero `estado_contrato` aun activo, resuelta con el criterio mas conservador (exclusion).

## 3. Indicadores de la restriccion de hierro (nucleo de OE-03)

Se derivaron de forma completamente independiente al companero:

| Indicador | Formula (fuente: anteproyecto 3.3) | N efectivo | Mediana |
|---|---|---|---|
| Desviacion de tiempo (dias) | plazo real − plazo pactado | 16.326 (85.1%) | +2 dias (mediana global) |
| Desviacion de costo (%) | (valor_pagado − valor_del_contrato) / valor_del_contrato × 100 | 8.681 (45.2%, solo con pago > 0) | 0% |
| Indice de cumplimiento de alcance | Categorico (total/parcial/no entregado) segun terminacion anticipada, suspension y cesion | 19.194 (100%) | 57.1% "Entregado en su totalidad" |

**Limitacion critica declarada:** la tasa de sobrecosto medida (practicamente nula, 0.1% de los contratos con dato valido) contrasta fuertemente con la cifra de referencia del problema de investigacion (~1 de cada 4 contratos, numeral 1.2 del anteproyecto). La hipotesis mas plausible, sin haber inspeccionado codigo excluido del companero, es que `valor_del_contrato` en el dataset integrado ya refleje el valor vigente tras adiciones formales, no el valor originalmente pactado — lo que subestimaria sistematicamente el sobrecosto medido contra `valor_pagado`. Esto se declara como limitacion metodologica de primer orden, no como hallazgo definitivo, y como recomendacion prioritaria de trabajo futuro (reconstruir el valor originalmente pactado, si es tecnicamente posible, a partir de un historial de adiciones en pesos).

## 4. EDA univariado y segmentado

- Las variables de tiempo, costo y valor del contrato son fuertemente asimetricas con outliers extremos heredados del dato fuente SECOP (tratados con winsorizacion P1-P99 solo para fines de visualizacion).
- **Retraso:** el 51.2% de los contratos con dato valido presentan plazo real mayor al pactado.
- La modalidad "Licitacion publica Obra Publica" muestra la mayor mediana de desviacion de tiempo (50 dias) de todas las modalidades, muy por encima de "Seleccion Abreviada de Menor Cuantia" (1 dia) y "Minima cuantia" (0 dias).
- Los consorcios/UT muestran mayor mediana de desviacion de tiempo (27 dias) que personas juridicas (1 dia) y personas naturales (0 dias), pero **menor** tasa de "No entregado" (12.4% vs. 26.4% en personas juridicas) — patron mixto, no unidireccional.
- Bogota D.C. (mayor volumen de contratacion) presenta la mediana de desviacion de tiempo mas alta (15 dias) entre los departamentos de mayor volumen.
- La asociacion entre modalidad de contratacion e indice de alcance es de tamaño **moderado** (V de Cramer = 0.268); entre tipo de contratista e indice de alcance es **debil-moderada** (V de Cramer = 0.184).

## 5. Contraste de hipotesis (H1, H2, H3)

| Hipotesis | Resultado | Interpretacion |
|---|---|---|
| **H1** — Licitacion publica vs. menor competencia | **Mixto.** Tiempo: NO se sostiene (licitacion con mayor desviacion, efecto -0.316). Costo: SI se sostiene (licitacion con menor desviacion, efecto 0.383). | La competencia en la seleccion parece disciplinar el costo final, no el cumplimiento del cronograma. Posible explicacion: los proyectos licitados publicamente tienden a ser de mayor envergadura y complejidad tecnica, lo que alarga los plazos independientemente del mecanismo de seleccion. |
| **H2** — Consorcios/UT vs. personas juridicas | **Parcial.** Se sostiene para N. total de modificaciones (efecto -0.372, moderado). No se sostiene para N. de adiciones en valor (efecto practicamente nulo pese a p-valor significativo). | Los consorcios modifican mas sus contratos en general (prorrogas, suspensiones, cesiones), pero no necesariamente mediante adiciones de valor monetario. |
| **H3** — NBI territorial vs. tiempo/costo | **No probada validamente** (sin datos de NBI municipal disponibles). Proxy exploratorio por departamento muestra heterogeneidad significativa (p<0.001 en ambas variables). | Limitacion de datos declarada explicitamente; se recomienda incorporar el NBI oficial del DANE por municipio como trabajo futuro antes de reformular esta prueba. |

## 6. Relacion con el marco teorico

Los hallazgos se interpretan bajo el marco de la restriccion de hierro (Atkinson, 1999; Meredith & Mantel, 2012; PMI, 2021), que postula que tiempo, costo y alcance son dimensiones interdependientes: una variacion en una afecta a las demas. El hallazgo de H1 (licitacion publica con mejor costo pero peor tiempo) es consistente con esa interdependencia: el mecanismo de seleccion que mas disciplina el costo (mayor competencia, mayor escrutinio tecnico) puede ir acompañado de proyectos mas complejos y, por tanto, de mayor riesgo de desviacion en el cronograma — un trade-off, no una mejora simultanea en las tres dimensiones. Esto es coherente con la literatura de antecedentes citada en el anteproyecto (Rodriguez Arevalo, 2021; Feigenbaum et al., 2024), que documenta que los factores asociados a sobrecostos y a retrasos no siempre coinciden.

## 7. Limitaciones generales declaradas (obligatorio, numerales 3.3 y 1.5.2 del anteproyecto)

1. La muestra analitica (19.194 contratos) es un subconjunto del universo integrado (48.331), sesgado deliberadamente hacia contratos con ejecucion avanzada o finalizada; los hallazgos no se extrapolan a contratos aun en ejecucion activa.
2. El indicador de desviacion de costo se calculo solo sobre el 45.2% de la muestra (contratos con `valor_pagado > 0`), por una brecha de reporte de SECOP no atribuible a este analisis.
3. El indice de alcance categorico no distingue "reduccion del objeto contratado" por ausencia de variable cruda especifica.
4. H3 no cuenta con datos reales de NBI municipal; se uso un proxy geografico exploratorio.
5. Las variables extrañas reconocidas por el anteproyecto (numeral 3.3: fenomenos climaticos, orden publico, cambios normativos, variaciones macroeconomicas) no fueron medidas ni controladas.
6. Ninguna asociacion o diferencia reportada implica causalidad; el diseño es transversal y censal condicionado (numeral 3.1 y 3.6.2).
7. La autoria del dato integrado de entrada (pipeline compartido, notebooks 00-06) sigue pendiente de confirmacion formal por parte del director (ver `MATRIZ_OBJETIVOS_CODIGO.md`, tarea de prioridad 1).

## 8. Insumo directo para el tablero interactivo (OE-04, fase posterior)

Este notebook y los notebooks 04-07 dejan listos, para su conexion directa a un tablero Streamlit + Plotly (decision ya aprobada en `MATRIZ_OBJETIVOS_CODIGO.md`, seccion 3, OE-04):
- Los tres indicadores continuos/categoricos de la restriccion de hierro (`analitica-descriptiva-andres/data/indicadores_restriccion_hierro.parquet`).
- Las segmentaciones por modalidad, tipo de contratista, orden y departamento (notebook 05).
- Las figuras estaticas ya generadas en `analitica-descriptiva-andres/figuras/`, reutilizables como referencia visual al disenar los componentes interactivos.

In [2]:
tabla_final_resumen = pd.DataFrame([
    {'Metrica': 'Contratos en universo integrado', 'Valor': 48331},
    {'Metrica': 'Contratos en muestra analitica (tras 3.6.3/3.6.4)', 'Valor': len(df)},
    {'Metrica': '% con retraso (desviacion tiempo > 0)', 'Valor': f"{(df['desviacion_tiempo_dias']>0).mean()*100:.1f}%"},
    {'Metrica': '% con sobrecosto (desviacion costo > 0, N=8681)', 'Valor': f"{(df['desviacion_costo_pct']>0).mean()*100:.1f}%"},
    {'Metrica': '% Entregado en su totalidad (indice de alcance)', 'Valor': f"{(df['indice_alcance']=='Entregado en su totalidad').mean()*100:.1f}%"},
    {'Metrica': 'H1 (licitacion vs. menor competencia)', 'Valor': 'Mixto: NO en tiempo, SI en costo'},
    {'Metrica': 'H2 (consorcios/UT vs. modificaciones)', 'Valor': 'Parcial: SI en N. modificaciones, NO en N. adiciones valor'},
    {'Metrica': 'H3 (NBI territorial)', 'Valor': 'No probada — falta dato NBI real'},
])
tabla_final_resumen


,Metrica,Valor
0,Contratos en universo integrado,48331
1,Contratos en muestra analitica (tras 3.6.3/3.6.4),19194
2,% con retraso (desviacion tiempo > 0),51.2%
3,"% con sobrecosto (desviacion costo > 0, N=8681)",0.1%
4,% Entregado en su totalidad (indice de alcance),57.1%
5,H1 (licitacion vs. menor competencia),"Mixto: NO en tiempo, SI en costo"
6,H2 (consorcios/UT vs. modificaciones),"Parcial: SI en N. modificaciones, NO en N. adi..."
7,H3 (NBI territorial),No probada — falta dato NBI real
